# Track 2 · 02 EEGNet — the organisers' classifier, on PROTEUS windows

**One job.** Train the organisers' EEGNet baseline on the warm-up data, check it learns,
score it the way the leaderboard does, and package it for submission.

Self-contained: upload this one file. The windows come from `track2_01_prepare` on Drive.

---

## The recipe, from source

Model and loss are `tracks/bci_decoding/solvers/eegnet.py`: `braindecode.models.EEGNet`,
cross-entropy, Adam at 1e-3. From the neuralbench `_motor_imagery_stream` config
(v0.4.0): **3 classes, label smoothing 0.1**, best validation balanced accuracy.

The warm-up contract is **streamed**: windows arrive one at a time (`B = 1`) in
chronological order within each participant x session, and each session runs on a fresh
copy of the model with `reset_state()` called first. Training is still ordinary batched
SGD; the streaming contract only binds the submission's `predict`.

## Scoring

**Balanced accuracy** per participant x session x interface (Graz / BrainHero) cell,
averaged equally. Three classes, chance is 33.3%. Our local split mirrors the stream
config's `SklearnSplit` (seed 33, 20% test / 20% valid). NeuralBench's public
references on this split: EEGNet 32.6%, CTNet 35.0%, REVE fine-tuned 36.0%.''')

## Setup

In [ ]:
ROOT = '/content/drive/MyDrive/Neurips26'

EPOCHS = 40
KEEP = 'best'           # best val epoch (val people are a real 20% of 10 subjects)
LR = 1e-3
LR_SCHEDULE = 'cosine'
BATCH = 64
DROPOUT = 0.5
F1 = 16
LABEL_SMOOTHING = 0.12  # soft targets (~92% / 4% / 4%) so the model is not pushed to be 100% sure; shared by all models
WEIGHT_DECAY = 0.0
N_SEEDS = 3
MODEL = 'reve_lora'     # 'eegnet', 'reve_probe' (frozen pretrained REVE), or 'reve_lora' (fine-tune)
CROP_S = 2.0            # keep only the first CROP_S s of each 4 s window: 2.0 = 1-3 s after the cue,
                        # before feedback starts at 3 s. None = the full contract window (1-5 s).
HEAD_ONLY_EPOCHS = 1    # reve_lora: adapters stay frozen for this many epochs, the head learns first
LR_ADAPTER, LR_HEAD = 2e-4, 1e-3   # reve_lora only; the other models use LR
LORA_RANK = 4           # reve_lora adapter rank (was 8); alpha is 2 x rank
LORA_DROPOUT = 0.10     # dropout on the input of each adapter branch (standard LoRA option); off at inference
LORA_RANK_FFN = 4       # LoRA on the two feed-forward matrices of every block (0 = off); weak: alpha = rank
LORA_ALPHA_FFN = 4      # scale alpha/rank = 1, half the attention adapters' 2
HEAD_KIND = 'perelec'   # 'perelec': per-electrode head (Track 1's); 'mqpool': K learned queries pool the
                        # electrodes; 'mean': average over electrodes and time (the head used until v13)
HEAD_SHRINK = 16        # perelec, single-layer version: width each electrode is shrunk to
HEAD_DIMS = (256, 128, 64, 32, 16)   # perelec funnel applied to each electrode's 512 REVE numbers,
                                     # halving each step (shared across electrodes); None = one
                                     # layer 512 -> HEAD_SHRINK
HEAD_DROPOUT_MID = 0.35 # dropout after each hidden layer of the funnel (HEAD_DROPOUT is used at the end)
MQ_K = 2                # mqpool: number of queries
HEAD_DROPOUT = 0.1      # reve_lora / reve_probe: dropout on the pooled features before the head

# Training-time augmentation, applied to every model (set a value to 0 to turn it off)
AUG_MASK_RATIO = 0.15   # share of electrode-tokens hidden per window, in irregular neighbouring patches
AUG_MASK_NOISE = 1.0    # 0 = clean blobs; higher = more scattered (1.0: neighbours hidden ~2x chance)
AUG_MASK_RADIUS = 0.05  # typical patch radius in metres (head radius ~0.095)
AUG_GAIN_SD = 0.1       # per-electrode random gain, log-normal sd (a quieter / noisier electrode)
AUG_SHIFT_S = 0.1       # random circular time shift of up to this many seconds

# Step 2b: leave-people-out check over all 10 people (slow: 5 folds x CV_SEEDS runs)
RUN_CV = True           # REVE LoRA folds take about an hour; EEGNet folds take minutes
CV_SEEDS = 2

# Fixed plot limits, so curves from different runs can be laid side by side
PLOT_LOSS_YLIM = (0.0, 2.4)    # the unregularised run reached 2.2 validation loss
PLOT_ACC_YLIM = (0.30, 0.60)

RUN_CONTROL = True
MAKE_ZIP = True
SEED = 33

HEAD_TAG = '-'.join(map(str, HEAD_DIMS)) if HEAD_DIMS else str(HEAD_SHRINK)
RUN_TAG = (f'proteus_{KEEP}_f{F1}_d{DROPOUT}_b{BATCH}_ls{LABEL_SMOOTHING}'
           f'_{LR_SCHEDULE}_x{N_SEEDS}_{MODEL}_crop{CROP_S}_r{LORA_RANK}-{LORA_DROPOUT}_ffn{LORA_RANK_FFN}_{HEAD_KIND}{HEAD_TAG}_hd{HEAD_DROPOUT}_e{EPOCHS}'
           f'_aug{AUG_MASK_RATIO}-{AUG_GAIN_SD}-{AUG_SHIFT_S}')
print(f'NOTEBOOK VERSION: track2_02 v17 PROTEUS   model={MODEL}   settings {RUN_TAG}   epochs {EPOCHS}')

In [ ]:
%pip install -q 'braindecode==1.8.1' huggingface_hub safetensors

import importlib
for mod in ('braindecode', 'torch', 'numpy', 'sklearn'):
    print(f'{mod:12} {importlib.import_module(mod).__version__}')

In [ ]:
import json, os, time, warnings, zipfile
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np, torch
from sklearn.metrics import balanced_accuracy_score

if ROOT.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

BASE = Path(ROOT) / 'Track2'
DATA = BASE / 'dreyer2026proteus'
WEIGHTS = BASE / 'weights'
WEIGHTS.mkdir(parents=True, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)


def load_npz(path, tries=5):
    """np.load that survives the Drive mount dropping mid-read (Errno 107): remount,
    then retry."""
    for attempt in range(tries):
        try:
            with np.load(path) as z:
                return {k: z[k] for k in z.files}
        except (OSError, zipfile.BadZipFile, EOFError) as e:
            if attempt == tries - 1 or not ROOT.startswith('/content/drive'):
                raise
            print(f'Drive read failed on {Path(path).name} ({e}); remounting and retrying')
            drive.mount('/content/drive', force_remount=True)
            time.sleep(5)


man = json.loads((DATA / 'manifest.json').read_text())
recs = man['recordings']
group = {r['subject']: r['group'] for r in recs.values()}
parts = {g: ([], [], [], []) for g in ('train', 'val', 'test')}   # X, y, person, session
CH = None
for name, r in sorted(recs.items()):
    z = load_npz(DATA / 'windows' / r['file'])
    p = parts[group[r['subject']]]
    p[0].append(z['X'].astype('float32')); p[1].append(z['y'])
    p[2].append(np.full(len(z['y']), r['subject']))
    p[3].append(np.full(len(z['y']), name))
    CH = CH or r['channels']
X, Y, P, S = {}, {}, {}, {}
for g, (xs, ys, ps, ss) in parts.items():
    X[g], Y[g], P[g], S[g] = (np.concatenate(xs), np.concatenate(ys), np.concatenate(ps),
                              np.concatenate(ss))
N_CHANS, N_TIMES = X['train'].shape[1:]
N_CLASSES = 3
SFREQ = man['preprocessing']['sfreq']
if CROP_S:
    # Windows on Drive are the harness's 4 s (1-5 s after the cue). The solver crops the
    # same way, because it already truncates inputs longer than the n_times it trained on.
    keep = int(round(CROP_S * SFREQ))
    assert keep <= N_TIMES, f'CROP_S={CROP_S} longer than the stored {N_TIMES} samples'
    X = {g: a[..., :keep] for g, a in X.items()}
    N_TIMES = keep
assert not set(P['test']) & set(P['train']) and not set(P['val']) & set(P['train'])
for g in ('train', 'val', 'test'):
    print(f'{g:5}  {len(set(P[g])):2d} people  {len(Y[g]):6,d} windows   '
          f'mi {(Y[g] == 0).sum():,} / sub {(Y[g] == 1).sum():,} / word {(Y[g] == 2).sum():,}')
print(f'{N_CHANS} channels, {N_TIMES} samples at {SFREQ:g} Hz, device {DEVICE}')


def scores(y, pred):
    return dict(bal_acc=float(balanced_accuracy_score(y, pred)), acc=float((y == pred).mean()))


def null(y, pred, n=200):
    """Balanced accuracy with the predictions shuffled across windows: same predictions,
    no link to the labels. A model has learned something only if it beats this."""
    return np.array([balanced_accuracy_score(y, rng.permutation(pred)) for _ in range(n)])

In [ ]:
from braindecode.models import EEGNet, REVE
import torch.nn.functional as F


def standardize(x):
    # Per-window per-channel standardization, matching the solver's _standardize().
    if not isinstance(x, torch.Tensor):
        x = torch.tensor(x)
    return (x - x.mean(dim=-1, keepdim=True)) / torch.sqrt(x.var(dim=-1, keepdim=True, unbiased=False) + 1e-8)


def train(Xtr, ytr, Xv, yv, epochs, label, quiet=False, seed=SEED):
    torch.manual_seed(seed)
    net = EEGNet(n_chans=N_CHANS, n_outputs=N_CLASSES, n_times=N_TIMES,
                 drop_prob=DROPOUT, F1=F1).to(DEVICE)
    opt = torch.optim.Adam(net.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = (torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
             if LR_SCHEDULE == 'cosine' else None)
    loss_fn = torch.nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    Xtr_t, ytr_t, Xv_t = torch.tensor(Xtr), torch.tensor(ytr), torch.tensor(Xv)
    gen = torch.Generator().manual_seed(seed)

    @torch.no_grad()
    def probs(Xa):
        net.eval()
        Xa = torch.as_tensor(Xa)
        return torch.cat([net(standardize(Xa[i:i + 512].to(DEVICE))).softmax(-1)
                          for i in range(0, len(Xa), 512)]).cpu().numpy()

    if not quiet:
        print(f'{label}: EEGNet {sum(p.numel() for p in net.parameters()):,} parameters, '
              f'{len(ytr):,} training windows, lr {LR_SCHEDULE}, seed {seed}')
    best, best_state, hist = -1.0, None, []
    for ep in range(1, epochs + 1):
        net.train()
        order = torch.randperm(len(ytr), generator=gen)
        tot, t0 = 0.0, time.time()
        for i in range(0, len(order), BATCH):
            b = order[i:i + BATCH]
            opt.zero_grad()
            loss = loss_fn(net(aug(standardize(Xtr_t[b].to(DEVICE)))), ytr_t[b].to(DEVICE))
            loss.backward(); opt.step()
            tot += float(loss) * len(b)
        lr_now = opt.param_groups[0]['lr']
        if sched is not None:
            sched.step()
        pv = probs(Xv_t)
        vl = float(-np.log(pv[np.arange(len(yv)), yv] + 1e-9).mean())
        sc = scores(yv, pv.argmax(1))
        hist.append(dict(epoch=ep, train_loss=tot / len(order), val_loss=vl, **sc))
        better = sc['bal_acc'] > best
        if better:
            best = sc['bal_acc']
            best_state = {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}
        if not quiet:
            print(f'  epoch {ep:2d}  train loss {tot / len(order):.4f}  val loss {vl:.4f}  '
                  f'val bal-acc {sc["bal_acc"]:.4f}  lr {lr_now:.1e}{"  *" if better else ""}  '
                  f'{time.time() - t0:.0f}s')
    if KEEP == 'best':
        net.load_state_dict(best_state)
    return net, hist, probs




def resample_fft(x, n_out):
    """Resample along time via the rfft spectrum (same function the solvers use)."""
    n_in = x.shape[-1]
    if n_in == n_out:
        return x
    spec = torch.fft.rfft(x.float(), dim=-1)
    out = torch.zeros(*spec.shape[:-1], n_out // 2 + 1, dtype=spec.dtype, device=x.device)
    keep = min(spec.shape[-1], out.shape[-1])
    out[..., :keep] = spec[..., :keep]
    return torch.fft.irfft(out, n=n_out, dim=-1) * (n_out / n_in)


def load_reve(n_classes):
    """Pretrained brain-bzh/reve-base, built the way Track 1 builds it (200 Hz, 1 s
    patches). The task head is dropped; ours is added by the caller."""
    net = REVE.from_pretrained('brain-bzh/reve-base', n_outputs=n_classes, n_chans=N_CHANS,
                               n_times=200, sfreq=200.0, chs_info=[{'ch_name': c} for c in CH])
    net.final_layer = torch.nn.Identity()
    bank = {k.casefold() for k in net._position_bank.mapping}
    missing = [c for c in CH if c.casefold() not in bank]
    print(f'REVE pretrained loaded; electrodes without a known position: {missing or "none"}')
    assert not missing, 'REVE would mis-size its input; fix the channel names first'
    return net


class ReveProbe(torch.nn.Module):
    """Frozen REVE encoder + a trainable linear head. No position bank needed: the
    head takes the pooled features and maps them to class logits."""

    def __init__(self, n_classes, device):
        super().__init__()
        self.net = load_reve(n_classes)
        self.net.final_layer = torch.nn.Identity()
        self.head = torch.nn.Linear(self.net.embed_dim, n_classes)
        self.drop = torch.nn.Dropout(HEAD_DROPOUT)
        self.to(device).eval()
        for p_ in self.net.parameters():
            p_.requires_grad_(False)

    def forward(self, x):
        x = resample_fft(x.float(), int(round(x.shape[-1] * 200.0 / SFREQ)))   # REVE is 200 Hz
        feats = self.net(x, return_features=True)['features']   # (B, ch, patch, embed)
        return self.head(self.drop(feats.mean(dim=(1, 2))))


def train_reve(Xtr, ytr, Xv, yv, epochs, label, quiet=False, seed=SEED):
    torch.manual_seed(seed)
    net = ReveProbe(N_CLASSES, DEVICE)
    opt = torch.optim.Adam(net.head.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = (torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
             if LR_SCHEDULE == 'cosine' else None)
    loss_fn = torch.nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    Xtr_t, ytr_t, Xv_t = torch.tensor(Xtr), torch.tensor(ytr), torch.tensor(Xv)
    gen = torch.Generator().manual_seed(seed)

    @torch.no_grad()
    def probs(Xa):
        net.eval()
        Xa = torch.as_tensor(Xa)
        return torch.cat([net(standardize(Xa[i:i + 512].to(DEVICE))).softmax(-1)
                          for i in range(0, len(Xa), 512)]).cpu().numpy()

    if not quiet:
        print(f'{label}: REVE probe {sum(p.numel() for p in net.head.parameters()):,} '
              f'trainable head params (encoder frozen), {len(ytr):,} windows, '
              f'lr {LR_SCHEDULE}, seed {seed}')
    best, best_state, hist = -1.0, None, []
    for ep in range(1, epochs + 1):
        net.train()
        order = torch.randperm(len(ytr), generator=gen)
        tot, t0 = 0.0, time.time()
        for i in range(0, len(order), BATCH):
            b = order[i:i + BATCH]
            opt.zero_grad()
            loss = loss_fn(net(aug(standardize(Xtr_t[b].to(DEVICE)))), ytr_t[b].to(DEVICE))
            loss.backward(); opt.step()
            tot += float(loss) * len(b)
        lr_now = opt.param_groups[0]['lr']
        if sched is not None:
            sched.step()
        pv = probs(Xv_t)
        vl = float(-np.log(pv[np.arange(len(yv)), yv] + 1e-9).mean())
        sc = scores(yv, pv.argmax(1))
        hist.append(dict(epoch=ep, train_loss=tot / len(order), val_loss=vl, **sc))
        better = sc['bal_acc'] > best
        if better:
            best = sc['bal_acc']
            best_state = {k: v.detach().cpu().clone() for k, v in net.head.state_dict().items()}
        if not quiet:
            print(f'  epoch {ep:2d}  train loss {tot / len(order):.4f}  val loss {vl:.4f}  '
                  f'val bal-acc {sc["bal_acc"]:.4f}  lr {lr_now:.1e}{"  *" if better else ""}  '
                  f'{time.time() - t0:.0f}s')
    if KEEP == 'best':
        net.head.load_state_dict(best_state)
    return net, hist, probs




class LoRALinear(torch.nn.Module):
    def __init__(self, base, r, alpha, dropout=0.0):
        super().__init__()
        self.base = base
        self.drop = torch.nn.Dropout(dropout)    # adapter branch only; the frozen base is untouched
        for p_ in self.base.parameters():
            p_.requires_grad_(False)
        self.A = torch.nn.Parameter(torch.zeros(r, base.in_features))
        self.B = torch.nn.Parameter(torch.zeros(base.out_features, r))
        torch.nn.init.kaiming_uniform_(self.A, a=5 ** 0.5)
        self.scale = alpha / r

    def forward(self, x):
        return self.base(x) + F.linear(F.linear(self.drop(x), self.A), self.B) * self.scale

    @property
    def weight(self):
        """Effective weight, W + scale * B A. braindecode >= 2.0 reads `net[1].weight` of the
        feed-forward GEGLU directly instead of calling the module, which would skip the
        adapter; with this property the adapter is applied there too. (The input dropout of
        this adapter does not apply on that path.)"""
        return self.base.weight + (self.B @ self.A) * self.scale



class PerElectrodeHead(torch.nn.Module):
    """Shrink each electrode's embedding with weights shared across electrodes (one linear
    layer, or a funnel of several), then a final layer with separate weights per electrode."""

    def __init__(self, n_chans, embed_dim, dims, n_outputs, dropout, mid_dropout=0.0):
        super().__init__()
        dims = [dims] if isinstance(dims, int) else list(dims)
        self.norm = torch.nn.LayerNorm(embed_dim)
        if len(dims) == 1:
            self.shrink = torch.nn.Linear(embed_dim, dims[0])        # keeps the older key names
        else:
            layers, prev = [], embed_dim
            for i, d in enumerate(dims):
                layers.append(torch.nn.Linear(prev, d))
                prev = d
                if i < len(dims) - 1:
                    layers += [torch.nn.GELU(), torch.nn.Dropout(mid_dropout)]
            self.shrink = torch.nn.Sequential(*layers)
        self.drop = torch.nn.Dropout(dropout)
        self.out = torch.nn.Linear(n_chans * dims[-1], n_outputs)

    def forward(self, tokens):                    # (B, chan, embed)
        z = self.drop(F.gelu(self.shrink(self.norm(tokens))))
        return self.out(z.flatten(1))


class MQPoolHead(torch.nn.Module):
    """K learned queries, each pooling the electrodes by a content-weighted average."""

    def __init__(self, k, embed_dim, n_outputs, dropout):
        super().__init__()
        self.queries = torch.nn.Parameter(torch.randn(k, embed_dim) / embed_dim ** 0.5)
        self.norm = torch.nn.LayerNorm(embed_dim)
        self.drop = torch.nn.Dropout(dropout)
        self.out = torch.nn.Linear(k * embed_dim, n_outputs)

    def forward(self, tokens):                    # (B, chan, embed)
        z = self.norm(tokens)
        w = torch.einsum('ke,bce->bkc', self.queries, z) / self.queries.shape[-1] ** 0.5
        w = w.softmax(dim=-1)
        pooled = torch.einsum('bkc,bce->bke', w, z)
        return self.out(self.drop(F.gelu(pooled.flatten(1))))


class ReveLoRA(torch.nn.Module):
    def __init__(self, n_classes, device, rank=LORA_RANK):
        super().__init__()
        self.net = load_reve(n_classes)
        self.net.final_layer = torch.nn.Identity()
        self.head_kind = HEAD_KIND
        E = self.net.embed_dim
        if HEAD_KIND == 'perelec':
            self.head = PerElectrodeHead(N_CHANS, E, HEAD_DIMS if HEAD_DIMS else HEAD_SHRINK, n_classes,
                                         HEAD_DROPOUT, HEAD_DROPOUT_MID)
        elif HEAD_KIND == 'mqpool':
            self.head = MQPoolHead(MQ_K, E, n_classes, HEAD_DROPOUT)
        else:
            self.head = torch.nn.Linear(E, n_classes)
        self.drop = torch.nn.Dropout(HEAD_DROPOUT)      # used by the 'mean' head only
        self.n_lora = 0
        for name, m in list(self.net.named_modules()):
            if isinstance(m, torch.nn.Linear) and name.split('.')[-1] in ('to_qkv', 'to_out'):
                parent = self.net.get_submodule(name.rsplit('.', 1)[0])
                setattr(parent, name.split('.')[-1], LoRALinear(m, rank, 2 * rank, LORA_DROPOUT))
                self.n_lora += 1
        self.n_ffn = 0
        if LORA_RANK_FFN:        # the two feed-forward matrices of each block, at net.1 and net.3
            for name, m in list(self.net.named_modules()):
                if (isinstance(m, torch.nn.Linear) and name.startswith('transformer.layers.')
                        and name.endswith(('.net.1', '.net.3'))):
                    parent = self.net.get_submodule(name.rsplit('.', 1)[0])
                    setattr(parent, name.split('.')[-1],
                            LoRALinear(m, LORA_RANK_FFN, LORA_ALPHA_FFN, LORA_DROPOUT))
                    self.n_ffn += 1
        for p_ in self.net.parameters():
            p_.requires_grad_(False)
        for n_, p_ in self.net.named_parameters():
            if n_.endswith(('.A', '.B')):
                p_.requires_grad_(True)
        self.to(device).eval()

    def forward(self, x):
        x = resample_fft(x.float(), int(round(x.shape[-1] * 200.0 / SFREQ)))   # REVE is 200 Hz
        feats = self.net(x, return_features=True)['features']      # (B, chan, patch, embed)
        if self.head_kind == 'mean':
            return self.head(self.drop(feats.mean(dim=(1, 2))))
        return self.head(feats.mean(dim=2))                        # average the time patches only

    def trainable_params(self):
        return ([p_ for n_, p_ in self.net.named_parameters() if p_.requires_grad]
                + list(self.head.parameters()))


def train_reve_lora(Xtr, ytr, Xv, yv, epochs, label, quiet=False, seed=SEED):
    torch.manual_seed(seed)
    net = ReveLoRA(N_CLASSES, DEVICE)
    adapters = [p_ for n_, p_ in net.net.named_parameters() if p_.requires_grad]
    opt = torch.optim.Adam([dict(params=adapters, lr=LR_ADAPTER),
                            dict(params=list(net.head.parameters()), lr=LR_HEAD)],
                           weight_decay=WEIGHT_DECAY)
    sched = (torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
             if LR_SCHEDULE == 'cosine' else None)
    loss_fn = torch.nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    Xtr_t, ytr_t, Xv_t = torch.tensor(Xtr), torch.tensor(ytr), torch.tensor(Xv)
    gen = torch.Generator().manual_seed(seed)

    @torch.no_grad()
    def probs(Xa):
        net.eval()
        Xa = torch.as_tensor(Xa)
        return torch.cat([net(standardize(Xa[i:i + 512].to(DEVICE))).softmax(-1)
                          for i in range(0, len(Xa), 512)]).cpu().numpy()

    if not quiet:
        n_ad = sum(p_.numel() for n_, p_ in net.net.named_parameters() if p_.requires_grad)
        print(f'{label}: REVE LoRA {n_ad:,} adapter params ({net.n_lora} attention + {net.n_ffn} feed-forward), '
              f'{HEAD_KIND} head, '
              f'{len(ytr):,} windows, lr {LR_SCHEDULE}, seed {seed}, head-only epochs {HEAD_ONLY_EPOCHS}, '
              f'lr adapters {LR_ADAPTER:g} / head {LR_HEAD:g}')
    best, best_state, hist = -1.0, None, []
    for ep in range(1, epochs + 1):
        for p_ in adapters:                      # staged: head alone first, then adapters too
            p_.requires_grad_(ep > HEAD_ONLY_EPOCHS)
        net.train()
        order = torch.randperm(len(ytr), generator=gen)
        tot, t0 = 0.0, time.time()
        for i in range(0, len(order), BATCH):
            b = order[i:i + BATCH]
            opt.zero_grad()
            loss = loss_fn(net(aug(standardize(Xtr_t[b].to(DEVICE)))), ytr_t[b].to(DEVICE))
            loss.backward(); opt.step()
            tot += float(loss) * len(b)
        lr_now = opt.param_groups[0]['lr']
        if sched is not None:
            sched.step()
        pv = probs(Xv_t)
        vl = float(-np.log(pv[np.arange(len(yv)), yv] + 1e-9).mean())
        sc = scores(yv, pv.argmax(1))
        hist.append(dict(epoch=ep, train_loss=tot / len(order), val_loss=vl, **sc))
        better = sc['bal_acc'] > best
        if better:
            best = sc['bal_acc']
            best_state = {k: v.detach().cpu().clone() for k, v in net.state_dict().items()
                          if k.endswith(('.A', '.B')) or k.startswith('head.')}
        if not quiet:
            print(f'  epoch {ep:2d}  train loss {tot / len(order):.4f}  val loss {vl:.4f}  '
                  f'val bal-acc {sc["bal_acc"]:.4f}  lr {lr_now:.1e}{"  *" if better else ""}  '
                  f'{time.time() - t0:.0f}s')
    if KEEP == 'best':
        net.load_state_dict(best_state, strict=False)
    return net, hist, probs


print('ready')

In [ ]:
# Source of truth for the mask generator is src/eeg_masking.py in the repo.
import numpy as np


class ElectrodeMasker:
    """positions: (C, 3) electrode coordinates in metres (REVE's position bank is in metres,
    head radius about 0.095). n_time: number of time tokens per window."""

    def __init__(self, positions, n_time=1, ratio=0.2, ratio_concentration=8.0,
                 spatial_scale=0.05, temporal_scale=1.0, scale_sigma=0.5,
                 n_bumps=(2, 5), noise=0.3, min_visible_frac=0.4, balance=True):
        pos = np.asarray(positions, dtype=float)
        self.C, self.T = len(pos), int(n_time)
        self.D = np.linalg.norm(pos[:, None] - pos[None], axis=-1)       # (C, C) metres
        self.ratio, self.conc = float(ratio), float(ratio_concentration)
        self.sp, self.tp, self.sig = float(spatial_scale), float(temporal_scale), float(scale_sigma)
        self.n_bumps, self.noise = n_bumps, float(noise)
        self.min_visible = max(1, int(round(min_visible_frac * self.C)))
        if balance:
            density = np.exp(-(self.D ** 2) / (2 * self.sp ** 2)).sum(1)
            w = 1.0 / density
        else:
            w = np.ones(self.C)
        self.centre_p = w / w.sum()

    def _one(self, rng):
        C, T = self.C, self.T
        a, b = self.ratio * self.conc, (1 - self.ratio) * self.conc
        ratio = float(rng.beta(a, b)) if 0 < self.ratio < 1 else self.ratio
        field = np.zeros((C, T))
        t_axis = np.arange(T)
        for _ in range(int(rng.integers(self.n_bumps[0], self.n_bumps[1] + 1))):
            c = rng.choice(C, p=self.centre_p)
            r = self.sp * np.exp(self.sig * rng.standard_normal())
            L = self.tp * np.exp(self.sig * rng.standard_normal())
            t0 = rng.uniform(-0.5, T - 0.5)
            amp = rng.uniform(0.5, 1.5)
            field += amp * np.exp(-self.D[c] ** 2 / (2 * r ** 2))[:, None] \
                * np.exp(-((t_axis - t0) ** 2) / (2 * L ** 2))[None]
        field = (field - field.mean()) / (field.std() + 1e-9)
        field = field + self.noise * rng.standard_normal((C, T)) + 1e-6 * rng.standard_normal((C, T))
        k = int(round(ratio * C * T))
        k = min(k, (C - self.min_visible) * T)
        mask = np.zeros(C * T, dtype=bool)
        mask[np.argsort(-field.ravel())[:k]] = True
        mask = mask.reshape(C, T)
        # every time token keeps at least `min_visible` electrodes
        for t in range(T):
            over = mask[:, t].sum() - (C - self.min_visible)
            if over > 0:
                hidden = np.flatnonzero(mask[:, t])
                mask[hidden[np.argsort(field[hidden, t])[:over]], t] = False
        return mask

    def __call__(self, batch_size, rng=None):
        """(B, C, T) boolean mask, True = hidden. Pass a numpy Generator for repeatability."""
        rng = rng if rng is not None else np.random.default_rng()
        return np.stack([self._one(rng) for _ in range(batch_size)])


def expand_mask(mask, patch_len, n_samples):
    """(B, C, T) token mask -> (B, C, n_samples) sample mask, each token covering patch_len."""
    out = np.repeat(mask, patch_len, axis=-1)[..., :n_samples]
    if out.shape[-1] < n_samples:                       # pad with the last token's value
        out = np.concatenate([out, np.repeat(out[..., -1:], n_samples - out.shape[-1], -1)], -1)
    return out


AUG_ON = AUG_MASK_RATIO > 0 or AUG_GAIN_SD > 0 or AUG_SHIFT_S > 0
if AUG_MASK_RATIO > 0:
    _tmp = load_reve(N_CLASSES)                    # only to read REVE's electrode positions
    _folded = {k.casefold(): k for k in _tmp._position_bank.mapping}
    POS = _tmp.get_positions([_folded[c.casefold()] for c in CH]).detach().cpu().numpy().astype(float)
    del _tmp
    TOK = int(SFREQ)                               # one time token = 1 s
    MASKER = ElectrodeMasker(POS, n_time=max(1, -(-N_TIMES // TOK)), ratio=AUG_MASK_RATIO,
                             spatial_scale=AUG_MASK_RADIUS, noise=AUG_MASK_NOISE)


def aug(x):
    """Applied to a standardised training batch (B, C, T). Hidden electrodes become zeros,
    which is also how the solver treats an electrode that is absent at inference."""
    if not AUG_ON:
        return x
    B, C, T = x.shape
    g = np.random.default_rng(int(torch.randint(0, 2 ** 31 - 1, (1,))))   # follows torch's seed
    if AUG_GAIN_SD > 0:
        x = x * torch.exp(AUG_GAIN_SD * torch.randn(B, C, 1, device=x.device))
    if AUG_SHIFT_S > 0:
        s = int(round(AUG_SHIFT_S * SFREQ))
        sh = g.integers(-s, s + 1, B).tolist()
        x = torch.stack([torch.roll(x[i], sh[i], dims=-1) for i in range(B)])
    if AUG_MASK_RATIO > 0:
        m = expand_mask(MASKER(B, g), TOK, T)
        x = x.masked_fill(torch.as_tensor(m, device=x.device), 0.0)
    return x


print(f'augmentation: mask {AUG_MASK_RATIO} (noise {AUG_MASK_NOISE}, radius {AUG_MASK_RADIUS} m), '
      f'gain sd {AUG_GAIN_SD}, shift {AUG_SHIFT_S} s' if AUG_ON else 'augmentation off')


## Step 1 — positive control

A pattern whose sign depends on the label is planted into a copy of the windows and the
same loop runs on it. If this does not learn, the loop is broken and Step 2 means nothing.

In [ ]:
if RUN_CONTROL:
    t_ax = np.arange(N_TIMES) / SFREQ
    pattern = (rng.standard_normal((N_CHANS, 1)) * np.sin(2 * np.pi * 10.0 * t_ax)).astype('float32')

    def plant(Xa, ya):
        return (Xa + 0.5 * (2 * ya - 1)[:, None, None] * pattern[None]).astype('float32')

    sub = rng.choice(len(Y['train']), min(4000, len(Y['train'])), replace=False)
    _, hist_c, _ = train(plant(X['train'][sub], Y['train'][sub]), Y['train'][sub],
                         plant(X['val'], Y['val']), Y['val'], 10, 'control', quiet=True)
    best_c = max(h['bal_acc'] for h in hist_c)
    print(f'control val balanced accuracy: epoch 1 {hist_c[0]["bal_acc"]:.3f} -> best {best_c:.3f}')
    print('CONTROL PASSES: the loop learns a planted signal' if best_c > 0.9 else
          'CONTROL FAILS: the loop cannot learn a planted signal; fix before Step 2')
else:
    print('control skipped')

## Step 2 — the real run

What to look for is validation balanced accuracy rising above its own null (the same
predictions shuffled across windows) and the best epoch coming after the first.

In [ ]:
members = []
for i in range(N_SEEDS):
    train_fn = {'reve_probe': train_reve, 'reve_lora': train_reve_lora}.get(MODEL, train)
    net_i, hist_i, probs_i = train_fn(X['train'], Y['train'], X['val'], Y['val'], EPOCHS,
                                      f'real, seed {i + 1}/{N_SEEDS}', seed=SEED + i)
    best_i = max(hist_i, key=lambda h: h['bal_acc']) if KEEP == 'best' else hist_i[-1]
    members.append(dict(net=net_i, hist=hist_i, probs=probs_i, best=best_i,
                        test=scores(Y['test'], probs_i(X['test']).argmax(1))))
    print(f'  -> seed {i + 1}: kept epoch {best_i["epoch"]} ({KEEP}), val bal-acc '
          f'{best_i["bal_acc"]:.4f}, test bal-acc {members[-1]["test"]["bal_acc"]:.4f}\\n')


def predict(Xa):
    return np.mean([m['probs'](Xa) for m in members], axis=0).argmax(1)


hist = members[0]['hist']
best = max(hist, key=lambda h: h['bal_acc'])
pv = predict(X['val'])
sv = scores(Y['val'], pv)
nullv = null(Y['val'], pv)
thr = float(np.percentile(nullv, 95))
climb = best['bal_acc'] - hist[0]['bal_acc']
print(f'averaged val bal-acc {sv["bal_acc"]:.4f} against shuffled predictions: mean '
      f'{nullv.mean():.4f}  sd {nullv.std():.4f}  95th pct {thr:.4f}')
TRENDING = bool(sv['bal_acc'] > thr and best['epoch'] > 1 and climb > 2 * nullv.std())
print('Trending: validation improved past epoch 1 and beats its null.' if TRENDING else
      'Above the null but no clear improvement after epoch 1.' if sv['bal_acc'] > thr else
      'Not trending: validation never beats its own null.')

pt = predict(X['test'])
st = scores(Y['test'], pt)
per_person = {str(s): float((pt[P['test'] == s] == Y['test'][P['test'] == s]).mean())
              for s in np.unique(P['test'])}
pp = np.array(list(per_person.values()))
seed_b = [m['test']['bal_acc'] for m in members]
print(f'\\ntest (the SklearnSplit test people)')
print(f'  {MODEL}, mean of {N_SEEDS} seeds   balanced accuracy {st["bal_acc"]:.4f}   '
      f'accuracy {st["acc"]:.4f}   (chance 0.333)')
print(f'  single seeds: {", ".join(f"{b:.4f}" for b in seed_b)}   '
      f'(spread {max(seed_b) - min(seed_b):.4f})')
print(f'  per person: min {pp.min():.2f}  median {np.median(pp):.2f}  max {pp.max():.2f}')

for i, m in enumerate(members):
    torch.save(m['net'].state_dict(), WEIGHTS / f'bci_eegnet_{RUN_TAG}_s{i}.pt')
(WEIGHTS / f'bci_eegnet_{RUN_TAG}.json').write_text(json.dumps(dict(
    run_tag=RUN_TAG, train_people=sorted(str(s) for s in set(P['train'])),
    check_people=sorted(str(s) for s in set(P['val'])), keep=KEEP, f1=F1, dropout=DROPOUT,
    batch=BATCH, label_smoothing=LABEL_SMOOTHING, lr_schedule=LR_SCHEDULE,
    n_seeds=N_SEEDS, test=st, per_person=per_person,
    members=[dict(best_epoch=m['best']['epoch'], test=m['test'], history=m['hist'])
             for m in members]), indent=1))
print(f'\\nsaved {N_SEEDS} members -> {WEIGHTS / f"bci_eegnet_{RUN_TAG}_s*.pt"}')

In [ ]:
import matplotlib
import matplotlib.pyplot as plt

chance_loss = float(np.log(N_CLASSES))
n = len(members)
fig, axes = plt.subplots(2, n, figsize=(5.2 * n, 7.4), squeeze=False)
for j, m in enumerate(members):
    h = m['hist']
    ep = [r['epoch'] for r in h]
    tl, vl = [r['train_loss'] for r in h], [r['val_loss'] for r in h]
    ax = axes[0, j]
    ax.plot(ep, tl, label='train loss (augmented batches)')
    ax.plot(ep, vl, label='validation loss')
    ax.axhline(chance_loss, ls='--', c='grey', lw=1, label=f'chance loss {chance_loss:.2f}')
    ax.axvline(m['best']['epoch'], ls=':', c='k', lw=1, label='kept epoch')
    ax.set_ylim(*PLOT_LOSS_YLIM)
    ax.set_title(f'seed {j + 1}: loss   (final gap {vl[-1] - tl[-1]:+.2f})')
    ax.set_xlabel('epoch'); ax.set_ylabel('loss'); ax.grid(alpha=0.3)
    if j == 0:
        ax.legend(fontsize=8, loc='upper right')
    ax = axes[1, j]
    ax.plot(ep, [r['bal_acc'] for r in h], c='C2', label='validation balanced accuracy')
    ax.axhline(1 / N_CLASSES, ls='--', c='grey', lw=1, label='chance')
    ax.axvline(m['best']['epoch'], ls=':', c='k', lw=1)
    ax.set_ylim(*PLOT_ACC_YLIM)
    ax.set_title(f'seed {j + 1}: validation balanced accuracy')
    ax.set_xlabel('epoch'); ax.set_ylabel('balanced accuracy'); ax.grid(alpha=0.3)
    if j == 0:
        ax.legend(fontsize=8, loc='upper left')
fig.suptitle(RUN_TAG, fontsize=9)
fig.tight_layout()
fig.savefig(WEIGHTS / f'curves_{RUN_TAG}.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'saved {WEIGHTS / f"curves_{RUN_TAG}.png"}   (same y-axis limits in every run; set in the first cell)')


## Step 2b — leave-people-out check (optional)

Trains on 8 of the 10 people and tests on the other 2, five times, so every person is tested once.
No best-epoch picking: each fold trains for exactly `EPOCHS` and keeps the last. The score is
balanced accuracy per session, averaged over sessions (our stand-in for the cell mean the
sealed phase uses). Set `RUN_CV = True` in the first cell. Use it to compare settings; the
single 2-person test split above is too noisy for that.


In [ ]:
if RUN_CV:
    people = sorted(set(P['train']) | set(P['val']) | set(P['test']))
    Xa = np.concatenate([X[g] for g in ('train', 'val', 'test')])
    Ya = np.concatenate([Y[g] for g in ('train', 'val', 'test')])
    Pa = np.concatenate([P[g] for g in ('train', 'val', 'test')])
    Sa = np.concatenate([S[g] for g in ('train', 'val', 'test')])
    order = np.random.default_rng(SEED).permutation(people)
    folds = [order[i::5] for i in range(5)]
    train_fn = {'reve_probe': train_reve, 'reve_lora': train_reve_lora}.get(MODEL, train)
    keep_before, KEEP = KEEP, 'last'
    fold_scores, single_scores = [], []
    for f, held in enumerate(folds):
        te = np.isin(Pa, held)
        probs_seeds = []
        for s_ in range(CV_SEEDS):
            net_f, _, probs_f = train_fn(Xa[~te], Ya[~te], Xa[te], Ya[te], EPOCHS,
                                         f'fold {f + 1}/5 seed {s_ + 1}', quiet=True, seed=SEED + s_)
            probs_seeds.append(probs_f(Xa[te]))
            del net_f
        def cell_mean(pr):
            pred, sess = pr.argmax(1), Sa[te]
            return float(np.mean([balanced_accuracy_score(Ya[te][sess == u], pred[sess == u])
                                  for u in np.unique(sess)]))
        ens, singles = cell_mean(np.mean(probs_seeds, 0)), [cell_mean(p_) for p_ in probs_seeds]
        fold_scores.append(ens); single_scores.append(np.mean(singles))
        print(f'  fold {f + 1}/5  people {sorted(str(h) for h in held)}  '
              f'session-mean bal-acc: ensemble {ens:.4f}, single seeds {np.round(singles, 4).tolist()}')
    KEEP = keep_before
    print(f'\nleave-people-out ({MODEL}, {CV_SEEDS} seeds per fold)')
    print(f'  ensemble   mean {np.mean(fold_scores):.4f}   sd over folds {np.std(fold_scores):.4f}')
    print(f'  single     mean {np.mean(single_scores):.4f}   (chance 0.333)')
else:
    print('Step 2b skipped (RUN_CV = False)')


## Step 3 — package for submission

The zip is one weights file per seed, the solver and a small config. The solver matches
incoming electrodes to the training ones **by name** and prints how many matched into the
Codabench log. Before zipping, it is loaded back the way the harness loads it and checked:
its predictions must equal this notebook's, and reordering or adding electrodes must not
change them.

In [ ]:
# Source of truth is solvers/track2_bci_eegnet.py in the repo.
SUBMISSION_PY = '"""Track 02 submission: EEGNet classifying cued mental commands.\n\nThe model follows the organisers\' `eegnet.py` baseline (braindecode EEGNet,\ncross-entropy). The weights come from `track2_02_eegnet.ipynb`, trained on\nPROTEUS (NEMAR nm000290) windows made with the harness\'s own preprocessing.\nA zip may hold several copies trained from different seeds; their class\nprobabilities are averaged before the argmax.\n\nThe streaming contract (warm-up and sealed): windows reach `predict` one at a\ntime (`B = 1`) in chronological order within each participant x session, and\neach session runs on a fresh `copy.deepcopy` of this model after calling its\noptional `reset_state()`. `reset_state` clears the optional running\nper-channel mean/variance kept by the per-window standardizer, which is the\nonly state this submission keeps.\n\nInputs are matched to the training electrodes by name: extra electrodes are\nignored, missing ones are filled with zeros, and it raises if none match. The\nclass count is fixed by training (three on the warm-up), so it also raises if\nthe harness asks for a different number of classes.\n"""\n\nimport json\n\nimport torch\nimport torch.nn.functional as F\n\nfrom benchmark_utils.base_solver import CompetSolver\nfrom braindecode.models import EEGNet\n\n\ndef resample_fft(x, n_out):\n    """Resample along time via the rfft spectrum. Pure torch, since nothing is\n    installed at evaluation."""\n    n_in = x.shape[-1]\n    if n_in == n_out:\n        return x\n    spec = torch.fft.rfft(x.float(), dim=-1)\n    out = torch.zeros(\n        *spec.shape[:-1], n_out // 2 + 1, dtype=spec.dtype, device=x.device\n    )\n    keep = min(spec.shape[-1], out.shape[-1])\n    out[..., :keep] = spec[..., :keep]\n    return torch.fft.irfft(out, n=n_out, dim=-1) * (n_out / n_in)\n\n\nclass ChannelMatchedClassifier:\n\n    def __init__(self, cfg, in_names, in_sfreq, device):\n        self.device = device\n        self.n_times = int(cfg["n_times"])\n        self.sfreq = float(cfg["sfreq"])\n        self.in_sfreq = float(in_sfreq)\n        self.nets = [\n            EEGNet(\n                n_chans=len(cfg["channels"]), n_outputs=int(cfg["n_classes"]),\n                n_times=self.n_times, drop_prob=float(cfg["dropout"]),\n                F1=int(cfg.get("f1", 8)),\n            ).to(device).eval()\n            for _ in cfg["weight_files"]\n        ]\n\n        lookup = {name.casefold(): i for i, name in enumerate(in_names)}\n        self.src = [lookup.get(name.casefold(), -1) for name in cfg["channels"]]\n        missing = [n for n, i in zip(cfg["channels"], self.src) if i < 0]\n        print(\n            f"[EEGNet-BCI] matched {len(self.src) - len(missing)}/{len(self.src)} "\n            f"training channels by name among {len(in_names)} inputs; "\n            f"zero-filled: {missing if missing else \'none\'}"\n        )\n        if len(missing) == len(self.src):\n            raise RuntimeError(\n                f"no training channel found in the input: {list(in_names)[:8]}"\n            )\n        # Running per-channel mean/variance over the current session, reset by\n        # reset_state(). Only set when the harness reports a channel count we\n        # can standardize (it always does: meta["n_chans"]).\n        self.adapt_mean = None\n        self.adapt_var = None\n        self.adapt_count = 0\n\n    def reset_state(self):\n        self.adapt_mean = None\n        self.adapt_var = None\n        self.adapt_count = 0\n\n    def _standardize(self, x):\n        # Per-window per-channel standardization, the training-time pipeline.\n        mean = x.mean(dim=-1, keepdim=True)\n        var = x.var(dim=-1, keepdim=True, unbiased=False)\n        return (x - mean) / torch.sqrt(var + 1e-8)\n\n    @torch.no_grad()\n    def predict(self, X):\n        X = torch.as_tensor(X, dtype=torch.float32).to(self.device)\n        out = torch.zeros(X.shape[0], len(self.src), X.shape[-1], device=self.device)\n        for dst, src in enumerate(self.src):\n            if src >= 0:\n                out[:, dst] = X[:, src]\n        # Match the training rate and window length. At the harness\'s own\n        # settings both are no-ops.\n        if self.in_sfreq != self.sfreq:\n            out = resample_fft(\n                out, int(round(out.shape[-1] * self.sfreq / self.in_sfreq))\n            )\n        if out.shape[-1] > self.n_times:\n            out = out[..., : self.n_times]\n        elif out.shape[-1] < self.n_times:\n            out = F.pad(out, (0, self.n_times - out.shape[-1]))\n        out = self._standardize(out)\n        probs = torch.stack([net(out).softmax(-1) for net in self.nets]).mean(0)\n        return probs.argmax(-1)\n\n\nclass Solver(CompetSolver):\n\n    name = "EEGNet-BCI"\n\n    requirements = ["pip::braindecode"]\n\n    def load_model(self, meta):\n        sub = meta["submission_dir"]\n        cfg = json.loads((sub / "bci_eegnet_config.json").read_text())\n        if meta.get("n_classes") not in (None, cfg["n_classes"]):\n            raise RuntimeError(\n                f"trained for {cfg[\'n_classes\']} classes, harness asks for "\n                f"{meta[\'n_classes\']}"\n            )\n        in_names = meta.get("ch_names") or [c["ch_name"] for c in meta["chs_info"]]\n        model = ChannelMatchedClassifier(cfg, in_names, meta["sfreq"], meta["device"])\n        for net, name in zip(model.nets, cfg["weight_files"]):\n            state = torch.load(\n                sub / name, map_location=meta["device"], weights_only=True\n            )\n            net.load_state_dict(state)\n        return model\n'


In [ ]:
# Source of truth for the REVE solver: solvers/track2_bci_reve_lora.py in the repo.
SUBMISSION_PY_REVE = '"""Track 02 submission: pretrained REVE with LoRA adapters and a head, averaged over seeds.\n\nThe encoder is `brain-bzh/reve-base`. Its weights ship once, in `encoder.pt`, and stay\nfrozen. Each seed ships only its LoRA adapters (rank r, on the 44 attention projections\n`transformer.layers.*.0.{to_qkv,to_out}`, and optionally a weak adapter on the two\nfeed-forward matrices of each block, `transformer.layers.*.1.net.{1,3}`) and its head, in\n`member_<i>.pt`. The head is per-electrode (Track 1\'s), a pooled-query head, or the plain\naverage, as the config says. The\nmembers share one copy of the encoder in memory, because the harness deep-copies the\nsubmitted model for every session.\n\nInput handling repeats the training notebook: electrodes matched to the training ones by\nname (extras ignored, missing ones zero-filled), the window cropped to the first\n`n_times` samples (the notebook\'s CROP_S), per-window per-channel standardisation, then\nresampling from the harness\'s 120 Hz to REVE\'s 200 Hz. The model keeps no state, so\n`reset_state` does nothing. Class probabilities are averaged over members before the\nargmax.\n\nThe model definition matches `track2_02_eegnet.ipynb`; keep the two in step.\n"""\n\nimport json\nimport os\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nfrom benchmark_utils.base_solver import CompetSolver\nfrom braindecode.models import REVE\n\nREVE_SFREQ = 200.0     # REVE\'s pretraining rate\nPATCH = 200            # REVE\'s patch_size in samples (1 s)\nLORA_TARGETS = ("to_qkv", "to_out")\nFFN_TARGETS = (".net.1", ".net.3")       # FeedForward.net is Sequential(norm, Linear, GEGLU, Linear)\n\n\ndef resample_fft(x, n_out):\n    """Resample along time via the rfft spectrum. Pure torch: nothing is installed at\n    evaluation."""\n    n_in = x.shape[-1]\n    if n_in == n_out:\n        return x\n    spec = torch.fft.rfft(x.float(), dim=-1)\n    out = torch.zeros(\n        *spec.shape[:-1], n_out // 2 + 1, dtype=spec.dtype, device=x.device\n    )\n    keep = min(spec.shape[-1], out.shape[-1])\n    out[..., :keep] = spec[..., :keep]\n    return torch.fft.irfft(out, n=n_out, dim=-1) * (n_out / n_in)\n\n\ndef standardize(x):\n    return (x - x.mean(-1, keepdim=True)) / torch.sqrt(\n        x.var(-1, keepdim=True, unbiased=False) + 1e-8\n    )\n\n\nclass LoRALinear(nn.Module):\n    """Frozen base plus a rank-r update. Names (`base`, `A`, `B`) match training."""\n\n    def __init__(self, base, r, alpha):\n        super().__init__()\n        self.base = base\n        self.A = nn.Parameter(torch.zeros(r, base.in_features))\n        self.B = nn.Parameter(torch.zeros(base.out_features, r))\n        self.scale = alpha / r\n\n    def forward(self, x):\n        return self.base(x) + F.linear(F.linear(x, self.A), self.B) * self.scale\n\n    @property\n    def weight(self):\n        """Effective weight, W + scale * B A. braindecode >= 2.0 reads `net[1].weight` of the\n        feed-forward GEGLU directly instead of calling the module, which would skip the\n        adapter; with this property the adapter is applied there too. (The input dropout of\n        this adapter does not apply on that path.)"""\n        return self.base.weight + (self.B @ self.A) * self.scale\n\n\nclass PerElectrodeHead(nn.Module):\n    """Shrink each electrode\'s embedding with weights shared across electrodes (one linear\n    layer, or a funnel of several), then a final layer with separate weights per electrode."""\n\n    def __init__(self, n_chans, embed_dim, dims, n_outputs, dropout=0.0, mid_dropout=0.0):\n        super().__init__()\n        dims = [dims] if isinstance(dims, int) else list(dims)\n        self.norm = nn.LayerNorm(embed_dim)\n        if len(dims) == 1:\n            self.shrink = nn.Linear(embed_dim, dims[0])        # keeps the older key names\n        else:\n            layers, prev = [], embed_dim\n            for i, d in enumerate(dims):\n                layers.append(nn.Linear(prev, d))\n                prev = d\n                if i < len(dims) - 1:\n                    layers += [nn.GELU(), nn.Dropout(mid_dropout)]\n            self.shrink = nn.Sequential(*layers)\n        self.drop = nn.Dropout(dropout)\n        self.out = nn.Linear(n_chans * dims[-1], n_outputs)\n\n    def forward(self, tokens):                    # (B, chan, embed)\n        z = self.drop(F.gelu(self.shrink(self.norm(tokens))))\n        return self.out(z.flatten(1))\n\n\nclass MQPoolHead(nn.Module):\n    """K learned queries, each pooling the electrodes by a content-weighted average."""\n\n    def __init__(self, k, embed_dim, n_outputs, dropout=0.0):\n        super().__init__()\n        self.queries = nn.Parameter(torch.randn(k, embed_dim) / embed_dim ** 0.5)\n        self.norm = nn.LayerNorm(embed_dim)\n        self.drop = nn.Dropout(dropout)\n        self.out = nn.Linear(k * embed_dim, n_outputs)\n\n    def forward(self, tokens):                    # (B, chan, embed)\n        z = self.norm(tokens)\n        w = torch.einsum("ke,bce->bkc", self.queries, z) / self.queries.shape[-1] ** 0.5\n        w = w.softmax(dim=-1)\n        pooled = torch.einsum("bkc,bce->bke", w, z)\n        return self.out(self.drop(F.gelu(pooled.flatten(1))))\n\n\ndef is_adapter_key(k):\n    return k.endswith((".A", ".B")) or k.startswith("head.")\n\n\nclass ReveLoRA(nn.Module):\n\n    def __init__(self, channels, n_classes, in_sfreq, rank, encoder_state,\n                 ffn_rank=0, ffn_alpha=0, head_kind="mean", shrink=16, mq_k=2, head_dims=None):\n        super().__init__()\n        self.head_kind = head_kind\n        self.in_sfreq = float(in_sfreq)\n        self.net = REVE(\n            n_outputs=n_classes, n_chans=len(channels), n_times=PATCH,\n            sfreq=REVE_SFREQ, chs_info=[{"ch_name": c} for c in channels],\n        )\n        self.net.final_layer = nn.Identity()\n        res = self.net.load_state_dict(encoder_state, strict=False)\n        if res.unexpected_keys:\n            raise RuntimeError(f"encoder.pt does not fit REVE: {res.unexpected_keys[:4]}")\n        for p in self.net.parameters():\n            p.requires_grad_(False)\n        names = [\n            n for n, m in self.net.named_modules()\n            if isinstance(m, nn.Linear) and n.split(".")[-1] in LORA_TARGETS\n        ]\n        for name in names:\n            parent = self.net.get_submodule(name.rsplit(".", 1)[0])\n            setattr(parent, name.split(".")[-1],\n                    LoRALinear(self.net.get_submodule(name), rank, 2 * rank))\n        if ffn_rank:                    # the feed-forward matrices, two per block\n            ffn_names = [\n                n for n, m in self.net.named_modules()\n                if isinstance(m, nn.Linear) and n.startswith("transformer.layers.")\n                and n.endswith(FFN_TARGETS)\n            ]\n            for name in ffn_names:\n                parent = self.net.get_submodule(name.rsplit(".", 1)[0])\n                setattr(parent, name.split(".")[-1],\n                        LoRALinear(self.net.get_submodule(name), ffn_rank, ffn_alpha))\n        E = self.net.embed_dim\n        if head_kind == "perelec":\n            self.head = PerElectrodeHead(len(channels), E, head_dims or shrink, n_classes)\n        elif head_kind == "mqpool":\n            self.head = MQPoolHead(mq_k, E, n_classes)\n        else:\n            self.head = nn.Linear(E, n_classes)\n\n    def tie_encoder_to(self, other):\n        """Share every frozen tensor with `other`, so ensemble members hold one encoder."""\n        theirs = dict(other.named_parameters())\n        for name, _ in list(self.named_parameters()):\n            if is_adapter_key(name):\n                continue\n            parent = self.get_submodule(name.rsplit(".", 1)[0])\n            setattr(parent, name.rsplit(".", 1)[1], theirs[name])\n\n    def forward(self, x):\n        x = resample_fft(x.float(), int(round(x.shape[-1] * REVE_SFREQ / self.in_sfreq)))\n        feats = self.net(x, return_features=True)["features"]     # (B, chan, patch, embed)\n        if self.head_kind == "mean":\n            return self.head(feats.mean(dim=(1, 2)))\n        return self.head(feats.mean(dim=2))                       # average the time patches only\n\n\nclass NameMatchedEnsemble:\n    """Reorders incoming electrodes into the trained order, zero-filling gaps, and\n    averages the members\' class probabilities."""\n\n    def __init__(self, members, channels, in_names, n_times, device):\n        self.members, self.device, self.n_times = members, device, int(n_times)\n        lookup = {n.casefold(): i for i, n in enumerate(in_names)}\n        self.src = [lookup.get(c.casefold(), -1) for c in channels]\n        missing = [c for c, i in zip(channels, self.src) if i < 0]\n        print(\n            f"[REVE-LoRA-BCI] matched {len(channels) - len(missing)}/{len(channels)} "\n            f"training channels by name among {len(in_names)} inputs; "\n            f"zero-filled: {missing if missing else \'none\'}; "\n            f"{len(members)} ensemble members"\n        )\n        if len(missing) == len(self.src):\n            raise RuntimeError(\n                f"no training channel found in the input: {list(in_names)[:8]}"\n            )\n\n    def reset_state(self):\n        pass                                       # stateless\n\n    @torch.no_grad()\n    def predict(self, X):\n        X = torch.as_tensor(X, dtype=torch.float32).to(self.device)\n        out = torch.zeros(X.shape[0], len(self.src), X.shape[-1], device=self.device)\n        for dst, src in enumerate(self.src):\n            if src >= 0:\n                out[:, dst] = X[:, src]\n        if out.shape[-1] > self.n_times:\n            out = out[..., : self.n_times]\n        elif out.shape[-1] < self.n_times:\n            out = F.pad(out, (0, self.n_times - out.shape[-1]))\n        out = standardize(out)\n        probs = torch.stack([m(out).softmax(-1) for m in self.members]).mean(0)\n        return probs.argmax(-1)\n\n\nclass Solver(CompetSolver):\n\n    name = "REVE-LoRA-BCI"\n\n    requirements = ["pip::braindecode"]\n\n    def load_model(self, meta):\n        sub = meta["submission_dir"]\n        os.environ["REVE_POSITIONS_PATH"] = str(sub)        # bundled; no download at eval\n        cfg = json.loads((sub / "reve_bci_config.json").read_text())\n        if meta.get("n_classes") not in (None, cfg["n_classes"]):\n            raise RuntimeError(\n                f"trained for {cfg[\'n_classes\']} classes, harness asks for "\n                f"{meta[\'n_classes\']}"\n            )\n        in_names = meta.get("ch_names") or [c["ch_name"] for c in meta["chs_info"]]\n        enc_state = torch.load(sub / "encoder.pt", map_location="cpu", weights_only=True)\n        members = []\n        for fname in cfg["member_files"]:\n            m = ReveLoRA(cfg["channels"], cfg["n_classes"], meta["sfreq"],\n                         cfg["rank"], enc_state,\n                         ffn_rank=cfg.get("ffn_rank", 0), ffn_alpha=cfg.get("ffn_alpha", 0),\n                         head_kind=cfg.get("head_kind", "mean"),\n                         shrink=cfg.get("head_shrink", 16), mq_k=cfg.get("mq_k", 2),\n                         head_dims=cfg.get("head_dims"))\n            state = torch.load(sub / fname, map_location="cpu", weights_only=True)\n            res = m.load_state_dict(state, strict=False)\n            absent = [k for k in res.missing_keys if is_adapter_key(k)]\n            if res.unexpected_keys or absent:\n                raise RuntimeError(f"{fname} mismatch: "\n                                   f"{(res.unexpected_keys or absent)[:4]}")\n            if members:\n                m.tie_encoder_to(members[0])\n            members.append(m)\n        members = [m.to(meta["device"]).eval() for m in members]\n        return NameMatchedEnsemble(members, cfg["channels"], in_names,\n                                   cfg["n_times"], meta["device"])\n'


In [ ]:
if MAKE_ZIP and MODEL != 'eegnet':
    print(f'Not packaging: this cell and its solver are EEGNet-only (MODEL={MODEL}). '
          'For REVE LoRA use Step 3b below.')
elif MAKE_ZIP and not (TRENDING or globals().get('FORCE_ZIP')):
    print('Not packaging: the run did not clear its null. Set FORCE_ZIP = True to override.')
elif MAKE_ZIP:
    import importlib.util, shutil, sys, types
    out = BASE / 'submissions' / f'track2_eegnet_{RUN_TAG}'
    if out.exists():
        shutil.rmtree(out)
    out.mkdir(parents=True)
    files = [f'weights_{i}.pt' for i in range(len(members))]
    for f_, m in zip(files, members):
        torch.save({k: v.cpu() for k, v in m['net'].state_dict().items()}, out / f_)
    (out / 'bci_eegnet_config.json').write_text(json.dumps(dict(
        channels=list(CH), n_times=int(N_TIMES), sfreq=SFREQ, n_classes=N_CLASSES,
        dropout=DROPOUT, f1=F1, weight_files=files,
        trained_on=f'PROTEUS, {len(set(P["train"]))} people, harness default preprocessing',
        test_bal_acc=st['bal_acc']), indent=2))
    (out / 'submission.py').write_text(SUBMISSION_PY)

    # Load it the way the harness does, with a stand-in for the base class.
    stub = types.ModuleType('benchmark_utils.base_solver')
    stub.CompetSolver = type('CompetSolver', (), {'name': 'x'})
    sys.modules.setdefault('benchmark_utils', types.ModuleType('benchmark_utils'))
    sys.modules['benchmark_utils.base_solver'] = stub
    spec = importlib.util.spec_from_file_location('_sub', out / 'submission.py')
    mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)

    def run(names, Xa, n_classes=N_CLASSES):
        m = mod.Solver().load_model({'ch_names': list(names), 'chs_info': None,
                                     'sfreq': SFREQ, 'submission_dir': out,
                                     'device': DEVICE, 'n_classes': n_classes})
        return m.predict(torch.tensor(Xa)).cpu().numpy()

    k = rng.choice(len(Y['test']), min(256, len(Y['test'])), replace=False)
    Xs = X['test'][k]
    base = run(CH, Xs)
    perm = rng.permutation(len(CH))
    shuffled = run([CH[i] for i in perm], Xs[:, perm])
    wide = run(list(CH) + ['EOG1', 'EMG1'],
               np.concatenate([Xs, rng.standard_normal((len(Xs), 2, N_TIMES)).astype('float32')], 1))
    keep_ch = [c for c in CH if c != 'Cz']
    part = run(keep_ch, Xs[:, [CH.index(c) for c in keep_ch]])
    try:
        run(CH, Xs[:4], n_classes=4)
        wrong_cls = False
    except RuntimeError:
        wrong_cls = True

    checks = [
        ('shape and labels', base.shape == (len(Xs),) and set(np.unique(base)) <= {0, 1, 2},
         f'{base.shape}, classes {sorted(set(base.tolist()))}'),
        ('same as this notebook', np.array_equal(base, pt[k]),
         f'{int((base != pt[k]).sum())} of {len(k)} differ'),
        ('order-independent', np.array_equal(base, shuffled), 'electrodes shuffled'),
        ('extra electrodes ignored', np.array_equal(base, wide), 'two unknown channels added'),
        ('missing electrode', part.shape == (len(Xs),), 'Cz zero-filled, still runs'),
        ('wrong class count raises', wrong_cls, 'asked for 4 classes: refuses rather than guessing'),
    ]
    for name, ok, detail in checks:
        print(f"  [{'ok  ' if ok else 'FAIL'}] {name}: {detail}")
    if all(c[1] for c in checks):
        zpath = shutil.make_archive(str(out), 'zip', out)
        print(f'\n{zpath}  ({Path(zpath).stat().st_size / 1e3:.0f} KB)')
        print('Download it from Drive and upload to Codabench (Track 2).')
    else:
        print('\nNot zipping: a check failed.')

## Step 3b — package the REVE LoRA ensemble

Same checks as Step 3, for `MODEL = 'reve_lora'`. The zip holds the frozen REVE encoder once
(about 270 MB), one small adapter-and-head file per seed, the solver, a config, and REVE's
electrode positions, since nothing is downloaded at scoring. It also streams 16 windows one
at a time to match the batched answer, as the harness will.


In [ ]:
if MAKE_ZIP and MODEL == 'reve_lora' and (TRENDING or globals().get('FORCE_ZIP')):
    import importlib.util, shutil, sys, types
    from braindecode.models import REVE
    assert len(members) == N_SEEDS, (
        f'members holds {len(members)} models, expected {N_SEEDS}: re-run Step 2 (the real run) '
        'to completion in this session before packaging')
    os.environ.pop('REVE_POSITIONS_PATH', None)    # a previous load pointed it at the folder deleted below
    out = BASE / 'submissions' / f'track2_{RUN_TAG}'
    if out.exists():
        shutil.rmtree(out)
    out.mkdir(parents=True)

    # The frozen pretrained encoder ships once; each seed ships only adapters and head.
    enc = REVE.from_pretrained('brain-bzh/reve-base', n_outputs=N_CLASSES, n_chans=N_CHANS,
                               n_times=200, sfreq=200.0, chs_info=[{'ch_name': c} for c in CH])
    torch.save({k: v.cpu() for k, v in enc.state_dict().items() if not k.startswith('final_layer')},
               out / 'encoder.pt')
    del enc
    files = [f'member_{i}.pt' for i in range(len(members))]
    for f_, m in zip(files, members):
        torch.save({k: v.cpu() for k, v in m['net'].state_dict().items()
                    if k.endswith(('.A', '.B')) or k.startswith('head.')}, out / f_)
    (out / 'reve_bci_config.json').write_text(json.dumps(dict(
        channels=list(CH), n_times=int(N_TIMES), sfreq=SFREQ, n_classes=N_CLASSES,
        rank=LORA_RANK, ffn_rank=LORA_RANK_FFN, ffn_alpha=LORA_ALPHA_FFN,
        head_kind=HEAD_KIND, head_shrink=HEAD_SHRINK, head_dims=list(HEAD_DIMS) if HEAD_DIMS else None,
        mq_k=MQ_K,
        member_files=files, crop_s=CROP_S,
        trained_on=f'PROTEUS, {len(set(P["train"]))} people, harness default preprocessing',
        test_bal_acc=st['bal_acc']), indent=2))
    (out / 'submission.py').write_text(SUBMISSION_PY_REVE)
    bank = list((Path.home() / 'mne_data').glob('*reve*positions*.json'))
    if not bank:
        import huggingface_hub
        bank = [Path(huggingface_hub.hf_hub_download('brain-bzh/reve-positions', 'positions.json'))]
    shutil.copy(bank[0], out / 'reve_positions.json')

    # Load it the way the harness does, with a stand-in for the base class.
    stub = types.ModuleType('benchmark_utils.base_solver')
    stub.CompetSolver = type('CompetSolver', (), {'name': 'x'})
    sys.modules.setdefault('benchmark_utils', types.ModuleType('benchmark_utils'))
    sys.modules['benchmark_utils.base_solver'] = stub
    spec = importlib.util.spec_from_file_location('_sub', out / 'submission.py')
    mod = importlib.util.module_from_spec(spec); spec.loader.exec_module(mod)

    def load(names, n_classes=N_CLASSES):
        return mod.Solver().load_model({'ch_names': list(names), 'chs_info': None,
                                        'sfreq': SFREQ, 'submission_dir': out,
                                        'device': DEVICE, 'n_classes': n_classes})

    def run(names, Xa, n_classes=N_CLASSES):
        return load(names, n_classes).predict(torch.tensor(Xa)).cpu().numpy()

    k = rng.choice(len(Y['test']), min(256, len(Y['test'])), replace=False)
    Xs = X['test'][k]
    base = run(CH, Xs)
    perm = rng.permutation(len(CH))
    shuffled = run([CH[i] for i in perm], Xs[:, perm])
    wide = run(list(CH) + ['EOG1', 'EMG1'],
               np.concatenate([Xs, rng.standard_normal((len(Xs), 2, N_TIMES)).astype('float32')], 1))
    keep_ch = [c for c in CH if c != 'Cz']
    part = run(keep_ch, Xs[:, [CH.index(c) for c in keep_ch]])
    streamer = load(CH)                      # one window per call, as the harness streams
    one_by_one = np.concatenate([streamer.predict(torch.tensor(Xs[i:i + 1])).cpu().numpy()
                                 for i in range(16)])
    try:
        run(CH, Xs[:4], n_classes=4)
        wrong_cls = False
    except RuntimeError:
        wrong_cls = True

    tol = max(1, int(0.01 * len(k)))        # GPU batching can flip a near-tie
    n_diff = int((base != pt[k]).sum())
    checks = [
        ('shape and labels', base.shape == (len(Xs),) and set(np.unique(base)) <= {0, 1, 2},
         f'{base.shape}, classes {sorted(set(base.tolist()))}'),
        ('same as this notebook', n_diff <= tol, f'{n_diff} of {len(k)} differ (allowed {tol})'),
        ('one window at a time', int((one_by_one != base[:16]).sum()) <= 1,
         f'{int((one_by_one != base[:16]).sum())} of 16 differ from the batched answer'),
        ('order-independent', np.array_equal(base, shuffled), 'electrodes shuffled'),
        ('extra electrodes ignored', np.array_equal(base, wide), 'two unknown channels added'),
        ('missing electrode', part.shape == (len(Xs),), 'Cz zero-filled, still runs'),
        ('wrong class count raises', wrong_cls, 'asked for 4 classes: refuses rather than guessing'),
    ]
    for name, ok, detail in checks:
        print(f"  [{'ok  ' if ok else 'FAIL'}] {name}: {detail}")
    if all(c[1] for c in checks):
        zpath = shutil.make_archive(str(out), 'zip', out)
        print(f'\n{zpath}  ({Path(zpath).stat().st_size / 1e6:.0f} MB)')
        print('Download it from Drive and upload to Codabench (Track 2).')
    else:
        print('\nNot zipping: a check failed.')
elif MAKE_ZIP and MODEL == 'reve_lora':
    print('Not packaging: the run did not clear its null. Set FORCE_ZIP = True to override.')
